In [6]:
!pip install -q transformers accelerate sentence-transformers torch


In [7]:
import torch
import numpy as np
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity


In [9]:
df = pd.read_parquet("/content/embedded_doc.parquet")

In [10]:
embed_model = SentenceTransformer("BAAI/bge-m3")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

In [11]:
model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

query -> embedding comparing with doc

In [15]:
def retrieve_chunks(query, top_k=3):
    query_embedding = embed_model.encode(query, normalize_embeddings=True)

    doc_embeddings = np.vstack(df["embedding"].values)

    similarities = cosine_similarity([query_embedding], doc_embeddings)[0]

    top_indices = similarities.argsort()[-top_k:][::-1]

    retrieved_chunks = df.iloc[top_indices]

    return retrieved_chunks


In [13]:
def build_mcp_prompt(query, retrieved_chunks):

    context_block = ""

    for i, row in enumerate(retrieved_chunks.itertuples(), 1):
        context_block += f"""
[Document {row.document_id} | Chunk {row.chunk_id}]
{row.text}
"""

    prompt = f"""
You are a document reasoning assistant.

Use ONLY the provided context to answer the question.
If the answer is not found in the context, say "Not found in provided documents."

----------------------
CONTEXT:
{context_block}
----------------------

QUESTION:
{query}

INSTRUCTIONS:
1. Identify relevant evidence.
2. Reason step-by-step internally.
3. Provide a clear final answer.

ANSWER:
"""
    return prompt


In [14]:
def generate_answer(prompt):

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            temperature=0.2,
            top_p=0.9,
            do_sample=True
        )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    return response


In [16]:
def ask_question(query):

    retrieved = retrieve_chunks(query, top_k = 3)

    prompt = build_mcp_prompt(query, retrieved)

    answer = generate_answer(prompt)

    return answer


In [19]:
query = "which paragraph and chunk ID contain text about chatbots"

response = ask_question(query)

print(response)



You are a document reasoning assistant.

Use ONLY the provided context to answer the question.
If the answer is not found in the context, say "Not found in provided documents."

----------------------
CONTEXT:

[Document 91a8061c-1cb8-4ca9-a891-46aaa8839ffe | Chunk 91a8061c-1cb8-4ca9-a891-46aaa8839ffe_chunk_27]
How ChatGPT Will Change Software Engineering Education ITiCSE 2023, July 8–12, 2023, Turku, Finland
using artificial intelligence virtual assistants. Computer Applications in Engineer-
ing Education (2022).
[11] Katharina Götz, Patric Feldmeier, and Gordon Fraser. 2022. Model-based Testing
of Scratch Programs. In 15th IEEE Conference on Software Testing, Verification
and Validation, ICST 2022, Valencia, Spain, April 4-14, 2022. IEEE, 411–421. https:
//doi.org/10.1109/ICST53961.2022.00047
[12] Miguel Goulão, Vasco Amaral, and Marjan Mernik. 2016. Quality in model-driven
engineering: a tertiary study. Software Quality Journal 24, 3 (2016), 601–633.
[13] Foteini Grivokostopoulou, 